# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "f87401db74ac11aa443118f6d0c2c9e6e1b6e49f558b86a120cdee8fee6fb1c4",
    "gzip_base64": "H4sIAAAAAAAC/60825LbRnbv8xWQ9gHADAhyRpZscQaqchTtZitZW2U7qVQxLFQTaJLIgAANgHOxoqr8Q/4wX5Jza6ABgpQcrx5GZF9O9zl97n2aL1++/P79h8nPjd47q/JQpDqdJGXR6KfG0U97XWU7XTS1o4rU2WlVHyqdBk6tdvtcT1SebQqdOnWTNck2KzbhxcVP+lDr2mm22km2OrmH7pWq62laHXbTzSFrVOU8ltW9rurQ+ZirYrqvypV2Ur1Wh7xxmtIpSicr1rrSRaLDix+0TmunOOz2z7BwktF/uNN1luvAWa93e70JnH1+4FX1Uwb7KTbO+3/9x++d6lA0gIKzLitnMtFPOjk0APTly5cX2W5fVo2jqs1eVbU237eq3ubZynz9z7oszOedarYX66rcOXv4BIMc6fiIHfK53h6aLG+/HVaAX6Lr+sI0ESqOApz27SiDDzbXa16DcA1rILHKzUJJWVU6Vw0inhVpvNfqvoOsc500VZbE60Oex7DxDcEDYuQaTub7Xz44kfPqZjabXfz044+/wBfcuBfHuHQc+2Gl6zJ/0J4fAknw3BfXy4uLCzgbQEvdvH7jIeL+/MKBf2m20XUTCb1CGeBT32PWbBk4TQjLvS48t1q5Pu2oqbTaMRT8h4ezysvkHs7dyRpdebnarVI154GwLZV617Obby7xjx+sXNef8/LhYZ8COTyazmtXujlUhWwv3Oon/gQ7Y0weK1gixnOlzQUPKj9owQkbom7ft41GyqrqOcLvIaIV14f1Onvy3LDZ7V1esh0WMnAUHw9XCFM47NqjJQI4MKBpdBOoPC8f40IV0Z9VXmv/yv2P4ghSpfe5SjRvRLaeq+fy0HhN2ag8cEROo7ezAFiz2mRF9Ab4QqU6Cl9/B/0HXUc/lAUwiy50tXmmL4IpbamOPAESMIAAZ/NOsrXjqeLZy4B/6kaBLHoPwaosc9+B4yrKxun1eFnRBOu8VI3fDkB5CbMaOBWI4j34dNAPeMi8ut9ygEy4nt1FsqG76M1sZtpnd7ivO94jNvKnu2v8LBMmN5emdeZ3vFWprNbOv+F6H6qqrDwXFYqZBCv+73//D6xUGxK+i65rJiLu05sJYXwkd6Whd1a7LYEGVOBzAUL4HUKXKHZ3EXURTtRwbn8/l4cq0c7uUDcOKEazQ6fWsOnUrP4n56+AwUYDLcpU5056qFSTlUUN207yA2xfg8reqzRFvez8oB9gqH5KEHlUk5X+FU6gQVKofUggd+op2x12ER0bHCVsRsjk06ZFQtKoQnXlMWGox78l6kQCAY4ChhkiYc/d9ZfxNviYo4F9OblWD2BNctCfsGtVADVqp2SYjjqkWSnkSGBLTXSN6wmpZS+gFGEd+OLdBIRYorPc46PiffpThOYLHLVXSdZkIBoLbL6CAQg0Q25YzAJaZ3K9ZLC0DWRq6q5UsdEejfCXHfa4seh6bkOWvfEgsEjQNFveOvsyzxIaYHroM4GBj8BqDkqwrA0tZpGBnDYn5bT5CjltOrHsJHN219wRzaZkRhDnBnHGXRAc/PAiqsEM6dSrdeNRz1lB/AdkI1B1DlroumV43kbgHIoMWBSNPcIkDwTwyICzkX/RusnZDw7gmg/JXlrvFAAtNiyGE6T4Ag6xHQBOyUY3kWm/YgZvp112Z7fIoH9aH3Zev22+9Duy5SVyn9eCE+YPePXh5N7UbRbtsqKbOlhZQDCTTTL/UkB3AIAb8vLdNpuPKL7SWYMLl63Az1gZ2qPLVw01AkvgVPRtnu2yxugdi1y40W0WIKp5GXBjjwoBYmP3gsDRbgOZyq1XNwMcyj1psmjB59Bwd5/pGNG7qDfkLtpmS5sUAmluczQKHO1dOoN7/Ryxw+E0c7WqvWZikBGZfI5c8OIegPfS2FDObWGiOA5WAGbN0C+poxy8UY8ZkwhydR3gVqdTsFIkOItxMi5Ht9wB7u/aYwMPsotY8xdWEzP/KpzNbi57eE379Ca9LniiZ8Lz40KrKuYZ7hCsu1I5apQ0RuHtaIGMG6o9uHspMrHfKTXTygv5F8PRxNnSvD0U960WJPEOPBWsfDx8Df6zrtDn+y3bk6AEJC0kgt0xgMKrmgAAEwfOAjVBVU+Mx2Z6haq9o4GxnxGQMp3QbCZTOwKiGC1BABsRM8VvnZaODISBQe2Ti67fkzvPAhdNRkzg3bmSrzDGna8CGLU/NKaTEZA2GgF/gt6ZWf9c8gJisyV3PrLNzz3/+JPLUVyMtHTnpKBcIk28rhQoZHfOhHIFuRh0UWyckLnxGy/6uyCN0Y0Sr9JFf6prxW+BawQpNhzizs0nWJMICMvQ/5/F/8VAICaz3wtEVKOCuorqNUcK5NWr/FE91/FNGv1SgRVJm+e9jlwyfK9uOg+url5EZNGAyRAOhjF7DWHPi+jG2L5i3xlHHOOH4MB7X/YxeQpwIoSyJcRd9//0W89nMYEKgBT80E5DnAxRQlyuPfD/C1ml64iwNQSR9Nyu0Q261j0EcSCwwafP/vEwF2Nx12/R7/rYUwCPl4cELgbs+D/H7BBwjXiqh/2ejTMdkwWtnlPUHzgEJXAMEBvvbrQJzCgtEIO8e12fIYC91yjiTc4FkkS4Y+MYi+6oJFLG5pgSD5aTwKfR7xnCE0SOAHL7OMijvr8LJZlmGLnXRDKLW1oJF+w9t8MqxLDUHG/4qB7MZ0oauP5AoA2+nmtjYYBwm0CRAaNg+Lg8l08q3sNXA6NNWuAYAdVrc/3PixHBMIkJ4HdMVng1hU2BAwEyaMyvi44p8SEh8mVgn7RIAZBSElGsduwImtk1Gufac8xDrqxs793b2Yhg/YXzZHYkBJYfJVQiuLfDcDBbD7eK0QLOoIhBnGfZyotWLkY4cQvETDtgCKZuuRNtMU11yiJ/lrWF8kx4zp5wk5VQCqiZh1jNNJ9bw919mgEbc+KJtTbl8uLyXvIkPLrK4CzBSmHiKZKsk6xHAzbop5DFi8l4Rgx/6rEtda94cMh5nBAcHl15ohI5eRcm5f4ZWcMw1RBkZz54+aP+F1Fvm+dMxY8y0KGZIP/oK6bO4xbTgczcmM1EDx353sS7EBkXBjNXMELhkeTAe1WURZYA2JYZ8OwQAkY00132hFDxFEPnL7x9CL0w70groRGpHnQq4Aw2cNa0J05pIgTwM5KtJHzR10VQE/B7wF3GzDDJ+wTl3SHNUXOqoUuLhtWh8BYup3FB9CdFWTdphpphQqoAqUTt+CdzA2Bic9zQAOJELRxWjHpHMCiBuTc4P5kr+G+f7OL6+k2ueS7R0l8GlK8mxmMas8RbPgcP7EJzFK02zjfqpDvrnW4UGveIMoHgeaS1R2LQjvXJYeF0od+L4cxcNuGMb8zc5gJ39dm+H3ycCbQNVCctdW1C/2RLh2cdMeWfkG+tmI8Q7u8qLR+LlVYQGgaLpX812DJocIyuqK8P5ShJwF6cNhGGR/6nhRUldyM6gwV78zRiwoHUd/4cvfpcFx4N8bn3yvQuh/aYnEAPvLr614r+38Gxcgb58vLGl/2gqESScLVgj+RLA8Qi4L13c01muga22qn4QVc1hhc3wHKqOdQQUaIs6tQNWlXBp9pXHYGcPVkqW9CPmb3HJYZFmGcD9grFWIUff/rw84dfAss4dR+P4Vrazd7MUOmFBVA9OB48itXRIsCMgN8PcvWDJgj4SVfTlslC571lDiXJWYtpr8XCcQaxl/n8K4ReGIWhj8vRxC0Z1qwAI/vMF1uHmlTlrweVY9uhgNPK1plOw68yr3amqe8ukLgPGofmj2LcqvxNF52lqskKx+1M92oAZNR6WbcycvrDpW+HBm4wIOCN9FSRgJKeFxGv8DWug7FlEIGiqcBFLY2C/L84xnQZfXJxaxAm0orMVq7oPrn2+cweWHeTYywhSR57luQl2v4wNojTeMSl3rrMU10FlhdNd0HSPu3iq3Fp4OUWrt209IdHjTeqCei+6Bw4wbQPUBqXvauHPedtY7oSzB503JSy325dyjHZl3cvIrOLc17JX9ptGXtgDhPA6RpEUGw+mO9+WIeLdHeGvbMdITIc4M6ixhE3dI40DDQJcBP32fZ3JEL4XW4uZj8zoBq7t7JujwVwBwtXTnZwsH/sQBiyOeKzd0PDbSOEKWpUcz4nDwP9jxjpYqdOBved3S5vyfxZjguhduStAN7W4ZFhQ/ektW0jlMc7f/Frmb8yurIqMRfV6F5IwZeuHBFMR+TLJr1Yxu4Ge3AMFij7BMSDehGNStu5o+g87K8VECHWwqRoli/Gg8eWgX0x1LRf8qRDTs0Nt91L253d9UdamgDyrk12EAzHGpiq5hjAijSOVaVFSkvJHgn7+DCLMe1OcXOEV8n8Psdkrm2lEVBLm+XAoJRYAXwqveNPcTRIiVhjbX42Um1BGfHGEW4kI9s1blvHniebNBsGl5h266l6JMhZBXerkgb8DhpnwxtYZtc/o+nGElMoX7u6BxJOHitjais5yKO6bfes1CfJHSacUnbbOBH5Wmp5mkol95iiQs4vqIf9I3GU5+5YVOaCfDDQvkQs2o4lmHyNGmRGmaUULxtiECt0DL4dBdnWDMVwUnvQ+q9hapcQJ9ZZDPPksEyj6vsY08OAYa43pfvZ9n76R9xmUlHJmXQOCBNewDIpaeQ9dD9QwH0fUL2DIWmIqr4WI2BNOD7tcf9yXKi/Jw4yJ8IZRHPWzi6rKcjrGMhmzbmNK67XcSPh1X5r2VciUBtIZ7twztdz3AnUX0TDZVFbuF8Mc98z4kMTOSCAzvHCvMNy3I8/Av6TgWdStJKQW6HrcEDuh2MGy3Yoqm6kpUaxnKZHmc5YioYFZh+ST8Si1fNGTOy6mX7YPYglzdQ/4GuOLsN3VABcpMq+uVr2r82O5rb3zJ0UAiCvBwnvu5aTMeD+F6ALDhuVEdjw2+MhAxXQYTHUDfYqA/sLYjzE/vSee3DE72AAfY4A+2KO7eztEjM6pdgK1ItTI2ZTs3d2CQecb25LYB1zn3Yo2MjWXlWWbIljyz20Ta+4ZJYb2Q7+Yjq882hubasdn7DWstT5IO2iF29HZ2ONC+umZp0UTd4VKRo/wA275UJMBbmmalFdcdUiNlo1iwgGq6SSew//BNzwLz++/+f4w7//l/Xth3+wYt6q3IAMy8WDWdq0cuB6NLjnhtsAev4KuZe9Xsqeg7HhG/tPn3v1luaeSVxSuW9d9nUs1hhQhtQwNd5iLyE4IMfN7H/Ns+NPPOwlDXu5nM9epZ/dfpEBzZNMf5vc7zKuli2CpWl3gtH8yOInKtmCj9IG6DJwAROXYymWgZ3j6cdRAgUI81F/HyeQ49r694Swe7xGz3/llWz/9Wj8vsqKBm0MGBLH5J4EOhZ6BOv8UG+ZULeSvNI9KH9yftJJiTV+iu6da9yrarCcFGz6usGOghJkVXXY41WOIRhnUcJ+uYlEY2mX4rZ5x8DNCinOErqFm7xcgcBwd3zZU2NdioxccRlk+99jRyUKMmOWBl7GfJ0lD9JvC0Lfvka2fZ2PXhD0Tkv2ZR/XbUcO6b1dwYL3Q5Zth7V5iuMULVsRNjT752ZbFnwlgOoXZBjOca2yHLyKNZYWJrAM1bLD2eG6z4b0x+nTw24F2nd21MH3SHJAIKrmcD7xDBZSv9UVc26+iq6PBa6lwnlgx1vgpLM5BjY1crPWAg3IVp72J8bvdrqqnd/lPgTyEIBvGUchX156n45c03nb8LmXEu7dtYq29b+kFlrMT2mGnkpjPWzmhMcZJrCtdmW7bQkC860fgkuj+ALpyrMr4Xt3JTezy2IPEre5nnl4AcOtPDy41pNrujURnwLiG042jwAaXrrQ9wNe33P6mct0oJWGv/nG7+Du8Y1JD/KuLEq+FWJgCng4uvYX8/l3y1DVCMqzQBk3HC9zcKp/9+Ybcrhhw/wdMPlm3hZPbH9z51Qa4OIzmDLNkqx5hng0/NyuPonwL68ud4ZJ1L7RIKiB/Emx+ghLIwJwRrdlCt/W4EEv2t1gDamAmCIFpEbYuw66IdCspJbQtAmb5eVjsM02W6mH/GY2m01vXs+44M1qe/XaJ3gqgQUlyYlvSMgTMw9KoHcBAOcIEK09tl2B5/doJ9wQCvX4HCguZsu7aPbV5FvhLQUzBcwlQEvEGRT4ba42ERpE3Pue763JF8BBVO4ME5bvIgRxGb59bUq3y2KdpeQMtsyW5NmesFGb5ZT2GMyCa39QCIdbJerAMFbkBtK7KHz9miV6BJdunClQozq3Rw10A2VqlyEr1Pt4rDl4xXt81zEL4Ns+m95wufIx69t7RFRKOBeEAgwe4GaWCAAMyqDV5N612sWPcKTlIzDLugkq3FSQk54JKv4v0egQBI9Z2rr8ooapY0Id0+nNLdZRUs8VNTFKEcJdUPMkH1O2WLE43sOcvopoUwKiOgmiOg0CuYFr9iCEpL0FN8SQq6PWkVz0n/GZB3gNsEgNHPlcgnOxKsHBk+SVuQd4VMHjKuodLtPsdpUjadTlo7paXT6uOMOgAg2ooR70A/xvhWHPmt9VkPrztLrU6kqvLvXKn94Y/q2qjnNZo42osUuYQfNX7POjkNCK/jvQX992zCqlZmAeI9DttA/crA8u19oX82VNmYVS6/GRzJrz6yHTVEYl2izFVF9RZ1wzDcJYqDz7TQu9MAzNc84ko4u2OmR5epCHJcC6TeQRwliNzMXItDP/ysPv3vUE1/HluUh1bE9ngxBWinsDQin45DLDWuWp9J2NvYt8GgMB4nS1hj6ghlZWsYhLXDgYsPKpOJftLOUP4Uvg0npdyMBz8JoDBlkQ9xCLZpQey/UDuHp8N0BDCfgUNR2cmthOXKpu3DmfPn72jUbht5Wxqir1XFOMHJDJq0VepVa6H8XdEneKLeCuXgXu0iq4F4vIU/0XUVvrUI9JzAfwYZ4leDQPNigrhTlRq5oLFdlvugKd5Y1dUYA4DhTeq5vuMQcrJ7lmLxyq86bt9VE38s/34SLtX+kHipYYlvfKBf2gvvdcFoZ3SatPC7QHVPM7yLyQugxWxru1asCXgd1G++2yRUTJhZqvllL8okZxma8mpxUk2oF+Kb39UsY6eQvfzl7wjuXtifm8vC3KHSYZRX338OkK8LUCvW3eioGSug5fS55xUCW+nL7yBy9ArAcU3kJWW3YXaMel40vaYnT2nYb9j5QKU0HAT3jDgXy94q/tc42bmeXQY7TdKEz4oh8EOtrZqnzNj56lSMW8VTw07ZMlelQIM7RKto5ON1bQ3VTZrieuQxIRfYylGFBokbDU0NOYtpl0aWvB35mz6rHIFa47Sh8qUb1iK9cC8e/Y5PekC0EsL46yK2Yb4++QkCs5/0Z1LHCERUMkcdKsTlSVWoJDniIykGf7NKwJmDHN52VgeTqJuDVjtPHt1zNP86fFK7x5QhW87AmsmBfMJdTkbloc8FcwvVjjGIgDIacPFiORQiO3hnM90IsqSrxk+H6ppHGFfiR2CIfC3mfGuRLJZ0QHvRP7MIwuUGOtRzplNR+wOdGqW2y5WE3GGGZ02ujIcTVG9LROFrMcPQzNY50RJKTc75X/JRwp3fVlVMabpWzwVY+lx69ch28LSHXuo2ofnYx+uAlr3dy5+xEtu12LJl6PohqUyaYqgb/Mry5Yt5J0g/WVt8C/d09/g5hyv8XCBnp2wFH3YI9ZAZF/xsXgoICfZTyvfQs+2kMG7J1sywq96mcpC86KTQ+HWs/zfQAb48CeecEPJMzHI7SUHZWEX9/MZpfkROfl5sar9gtEaTnN5QO5j+YLaTAzZOAUS5zT1KZGkzLWUWZ0v1G7kXxnN5I9KOQSdmMixU3oEHLD6sTzLPI/CbEIUSZvk79We0bYOIiMKP09AWunCr65RCJHHqGxOOGWLp07Z/KKwvNVbYaedk2X/rs3p96X9f6hQqVfBRk+McBlqAdiEUw+nECByy0Fh1gqr9IzyTipDv2ZfwtjDWF67aQlra0xC4U1CSwu0ybbrSodOh+wmHOyx/LIrtp5c8D0SaPBZ92qWlt8HVoGh1wm83aPqNYlSqwADTQV5mMAZXnWIA4CXj1G7PAE4dvXU5zGtzPwgTnxOrSjGZ5+ifMgkqHV3Tn9F7iYUceaWa1SwG/H95pzGoonSd8paePO8e/J94GOEXChYZqpDVbaZwk/5bjFX11Rh6YE+QJZTnKV7eihPUqxo5LkUKnkOXCYvEi4mqQ6eab71X4BbegOohes8f8jd4v9S0O6KMWq0NHLw7FbM2/8pm2k4IetBvjLt+Yaxvadv3hpBoge30iN3KCN3lCJ7wQwxmsIz1+0jUVrclFsX1l15WhtfVMvLQ39owlpJoyRCutRBN1k+y25zAgCxH12WGgOghlDp66JQeURUHsPMVL7KFNkKP/QT33bvlVxssbcmyig4xYEn1dxraMN5PrpdFh9/CRJLqXX/OMuptR47X5qGfEzV+TzCmQs6sOKX5x+fP+3+OYbdxyGu8ue6MEczw9fX3rht5c2cbnO70oeH5wBjexZU41s4SzajQXtCjaX/sHXN30CMO5++7gmz1a7/SvwrTVOWVHb9dub+/G5MNQdeXpD2GCMgLEqndhCFKOFBud4JGzC3sUwDbQchJXHZ0ghxSea3BMn60DFJ5WU6CsGOXxywr1X0rs8d1Lt4bMwM3o1KCOb27o0SCdIJhfSVz6Bkbzh43d5+O6NJB3OXI1Zv9+gi02zJUv2XdgH08ur2ICHMEYq9PtYIyeMaDMp4e9Zs088VSr5ArteJjY0ALtp1dcMrCEaTPI48LdDxJPrEobHyIzdO3YwzEQmU2AXVzFy7ty2QHCKp2tNreLL/69s4mtVehpHv1xgPZU7JoQ7abiPt/67Rfe0EYpZ/8asXU8Jt9zZm/dR8hyqtQjjz5LaKCfQT0xFSiVGlgyO+D5tzW40oq4tBjLvZU7qdrFiI/zM3iLzK2PWS1Vzk7hD+HsyJrG4j8wvz4XfVxvC8SN+q7wUbHWVURlsFMdpmcSxfwt8EYG3maYxcggNrHFkEwHP7iAqSHF5y5O2XrHBDJrJ0zyXyxKpxJ/alazvuZMJWxw3IMWFpQkDqOOzSix9PjWlexI3nGZ+ussIE4MgFzuQHwSM3s5OrMkpsrNz35xZG2O3s5PD19+dWBnf+1nonlnE/u0FcKWzBNN1J35hYdmuLL/ScBpsW3gw4bqE4V7QeiDvkjPg0sNbWAddv0DkrOcODBkEpwLHHS1LfHP+mI0fCxBeGJkGpXyMQVn2OMagjr9LOLa0FGyASaZidASNigaIq92jlTkNDDiPLNxWd/Li+GtY7cG8mQVvZ8H1zWw5ZPpjOGwjxvhGrqpgcA0SSyTFmXX3VAi/hSK1ET+Wce03hDWCMb9rQIOlWoY+g6wFAoF/l0WYmBv7iWNus+9+ZCq+WLX0LDVab0CH7xipf/iY0CqnHiDExLfqDc+EejS3C9+6MsGuvJDgxZ9opDk/Dnpuv6aikLM7XWnUyZIoGnhUEUWtdkHU7yk6tQ70S2992vooJjYWSZ2vgA3OYiAFVcw+dF/aFkv1CNnWQlGrfOnb0r9XLdQYr6BS6l/nWduYC+msOuWOOCOMQ1nFOOBZQXyay2igALdyFINRXJZp/Y6nADY/5Em1SbDlmByFOAZ04hhtfBzjTx+hrb/4PxYbJvdhVwAA"
  },
  "electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
